# MPO - Vybrané ukazatele zpracovatelského průmyslu

This notebook ingests financial indicators data from **MPO** (Ministerstvo průmyslu a obchodu / Ministry of Industry and Trade) into the bronze layer.

## Source
- **Location:** `/Volumes/agentbricks/sector_data_raw/mpo_data/vybrane-ukazatele.csv`
- **Format:** CSV (semicolon-separated, windows-1250 encoded, with Excel `=""` wrapping)
- **Content:** 63 financial indicators across 108 CZ-NACE industry sectors, yearly data from 2008 to 2024

## Table Created

| Table | Description |
| --- | --- |
| `agentbricks.sector_data_bronze.mpo_industry_indicators` | Financial indicators by CZ-NACE sector and year (unpivoted to long format) |

## Columns
- `nace_code` — CZ-NACE industry classification code (e.g. "C", "10.1", "29")
- `indicator` — Financial indicator name (e.g. "Aktiva celkem [v tis. Kč]", "EBIT [v tis. Kč]")
- `year` — Reporting year (2008–2024)
- `value` — Numeric value of the indicator

In [0]:
# Configuration
SOURCE_PATH = "/Volumes/agentbricks/sector_data_raw/mpo_data/vybrane-ukazatele.csv"
TARGET_TABLE = "agentbricks.sector_data_bronze.mpo_industry_indicators"

In [0]:
import pandas as pd
import re

# Read CSV with correct encoding and separator
df = pd.read_csv(SOURCE_PATH, encoding='windows-1250', sep=';', quotechar='"', dtype=str)

# Clean ="..." Excel artifacts from column names
df.columns = [re.sub(r'^="(.*)"$', r'\1', col) for col in df.columns]

# Clean ="..." artifacts from string columns (CZ-NACE and HODNOTY)
for col in ['CZ-NACE', 'HODNOTY']:
    df[col] = df[col].apply(lambda x: re.sub(r'^="(.*)"$', r'\1', str(x)) if pd.notna(x) else x)

# Rename for clarity
df = df.rename(columns={'CZ-NACE': 'nace_code', 'HODNOTY': 'indicator'})

# Unpivot year columns from wide to long format
year_columns = [c for c in df.columns if c.isdigit()]
df_long = df.melt(
    id_vars=['nace_code', 'indicator'],
    value_vars=year_columns,
    var_name='year',
    value_name='value_raw'
)

# Parse numeric values: remove space thousand separators, handle edge cases
def parse_value(val):
    if pd.isna(val) or val == '' or val == 'x' or val == '-':
        return None
    # Remove spaces (thousand separators)
    cleaned = str(val).replace('\xa0', '').replace(' ', '').replace(',', '.')
    try:
        return float(cleaned)
    except ValueError:
        return None

df_long['value'] = df_long['value_raw'].apply(parse_value)
df_long['year'] = df_long['year'].astype(int)

# Drop rows with null values
df_long = df_long[df_long['value'].notna()].copy()
df_long = df_long[['nace_code', 'indicator', 'year', 'value']]

print(f"Rows after cleaning: {len(df_long):,}")
print(f"Distinct NACE codes: {df_long['nace_code'].nunique()}")
print(f"Distinct indicators: {df_long['indicator'].nunique()}")
print(f"Year range: {df_long['year'].min()} - {df_long['year'].max()}")
print(f"\nSample:")
print(df_long.head(10).to_string())

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Define schema
schema = StructType([
    StructField("nace_code", StringType(), False),
    StructField("indicator", StringType(), False),
    StructField("year", IntegerType(), False),
    StructField("value", DoubleType(), False),
])

# Create Spark DataFrame
spark_df = spark.createDataFrame(df_long, schema=schema)

print(f"Spark DataFrame: {spark_df.count():,} rows")
spark_df.printSchema()
display(spark_df.limit(10))

In [0]:
# Write to Delta table
spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TARGET_TABLE)

print(f"\u2705 Table written: {TARGET_TABLE}")
print(f"   Rows: {spark.table(TARGET_TABLE).count():,}")

# Add table comment
spark.sql(f"""
    COMMENT ON TABLE {TARGET_TABLE} IS
    'Annual financial indicators for Czech manufacturing sectors by CZ-NACE classification. Source: MPO (Ministerstvo pr\u016fmyslu a obchodu). Indicators include assets, liabilities, revenues, profits, employment, and financial ratios from 2008-2024.'
""")

# Add column comments
spark.sql(f"ALTER TABLE {TARGET_TABLE} ALTER COLUMN nace_code COMMENT 'CZ-NACE industry classification code (e.g. C=manufacturing, 10=food, 29=motor vehicles)'")
spark.sql(f"ALTER TABLE {TARGET_TABLE} ALTER COLUMN indicator COMMENT 'Financial indicator name with unit (e.g. Aktiva celkem [v tis. K\u010d])'")
spark.sql(f"ALTER TABLE {TARGET_TABLE} ALTER COLUMN year COMMENT 'Reporting year'")
spark.sql(f"ALTER TABLE {TARGET_TABLE} ALTER COLUMN value COMMENT 'Numeric value of the indicator (in thousands CZK for monetary, headcount for employment, ratio for financial ratios)'")

print("   Table and column comments added.")

In [0]:
from pyspark.sql import functions as F

result_df = spark.table(TARGET_TABLE)

print("=" * 60)
print("VALIDATION")
print("=" * 60)
print(f"Total rows: {result_df.count():,}")
print(f"Distinct NACE codes: {result_df.select('nace_code').distinct().count()}")
print(f"Distinct indicators: {result_df.select('indicator').distinct().count()}")
print(f"Year range: {result_df.agg(F.min('year')).collect()[0][0]} - {result_df.agg(F.max('year')).collect()[0][0]}")

print("\n=== Sample data (NACE=C, first 5 indicators, 2024) ===")
display(
    result_df
    .filter((F.col("nace_code") == "C") & (F.col("year") == 2024))
    .orderBy("indicator")
    .limit(10)
)

print("\n=== Top NACE codes by number of records ===")
display(
    result_df.groupBy("nace_code")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
)